In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "XRPUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,dow_cos,dom_sin,dom_cos,month_sin,month_cos,macd,macd_signal,macd_hist,atr_14,atr_norm
0,2025-09-01 00:00:00+00:00,2.7757,2.7757,2.7723,2.7757,83483.8,2025-09-01 00:00:59.999999+00:00,2.315646e+05,1015,58143.3,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000000,0.000000,0.000000,NaN,NaN
1,2025-09-01 00:01:00+00:00,2.7757,2.7772,2.7753,2.7772,84999.6,2025-09-01 00:01:59.999999+00:00,2.359481e+05,696,62509.2,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000034,0.000019,0.000015,NaN,NaN
2,2025-09-01 00:02:00+00:00,2.7772,2.7774,2.7756,2.7767,36682.2,2025-09-01 00:02:59.999999+00:00,1.018428e+05,582,21315.5,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,0.000027,0.000022,0.000005,NaN,NaN
3,2025-09-01 00:03:00+00:00,2.7766,2.7770,2.7751,2.7751,28222.0,2025-09-01 00:03:59.999999+00:00,7.834349e+04,843,11713.8,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.000034,0.000003,-0.000037,NaN,NaN
4,2025-09-01 00:04:00+00:00,2.7751,2.7751,2.7605,2.7629,838170.8,2025-09-01 00:04:59.999999+00:00,2.318988e+06,4641,203901.1,...,1.0,0.201299,0.97953,-1.0,-1.836970e-16,-0.000552,-0.000162,-0.000390,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 284,601
[info] optuna train rows: 182,144
[info] valid rows:        45,536
[info] test rows:         56,921


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-19 23:24:52,108] A new study created in memory with name: no-name-eff0e61b-f182-409f-b3fb-e14cf725e232


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:05<?, ?it/s]

Best trial: 0. Best value: -0.00261801:   0%|          | 0/50 [00:05<?, ?it/s]

Best trial: 0. Best value: -0.00261801:   2%|▏         | 1/50 [00:05<04:11,  5.13s/it]

...

Best trial: 8. Best value: 0.0242694:  98%|█████████▊| 49/50 [04:01<00:01,  1.82s/it]

Best trial: 8. Best value: 0.0242694:  98%|█████████▊| 49/50 [04:01<00:01,  1.82s/it]

Best trial: 8. Best value: 0.0242694: 100%|██████████| 50/50 [04:01<00:00,  1.79s/it]

Best trial: 8. Best value: 0.0242694: 100%|██████████| 50/50 [04:01<00:00,  4.83s/it]

[I 2026-03-19 23:28:53,716] Trial 49 finished with value: 0.02151967975944134 and parameters: {'n_estimators': 600, 'max_depth': 3, 'learning_rate': 0.0021148881119178444, 'subsample': 0.6545241502942464, 'colsample_bytree': 0.7556278721689292, 'min_child_weight': 16, 'reg_alpha': 0.005703967519695986, 'reg_lambda': 1.9112214472757804e-07}. Best is trial 8 with value: 0.024269445235060336.

[optuna] best trial
value: 0.024269
params:
  n_estimators: 400
  max_depth: 3
  learning_rate: 0.001607773799018405
  subsample: 0.6519324326265077
  colsample_bytree: 0.8144883267916221
  min_child_weight: 4
  reg_alpha: 1.6096877119551739e-06
  reg_lambda: 9.015642219760761e-05


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final xgb...


[training] done in 53.93s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...

===== RESULTS =====
Train IC:      0.461367
Test IC:       -0.008258
Train Rank IC: 0.035685
Test Rank IC:  0.003015
Train RMSE:    0.002728
Test RMSE:     0.002393


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
volume_mom_5        0.090325
trades_z            0.086755
imbalance_5         0.068836
num_trades_mom_5    0.064300
vol_ratio_5_30      0.061810
volume_z            0.056378
trend_strength      0.055139
trend_x_imb         0.055056
imbalance           0.048810
range_ratio         0.041293
dow_sin             0.034043
mom_3               0.033478
imbalance_15        0.029559
dist_ma_5           0.027277
vol_5               0.024995
dist_ma_15_z        0.023716
mom_15              0.023243
dom_cos             0.022532
dom_sin             0.019450
mom_x_imb           0.016792
vol_regime_ratio    0.015638
mom_60              0.014491
vol_30              0.012252
dist_ma_15          0.007724
mom_30              0.007531
macd_hist           0.006993
mr_x_vol            0.006269
atr_norm            0.006029
vol_15              0.005899
range_15            0.005782
mom_10              0.004954
dist_ma_30          0.004677
mom_5               0.004269
bar_range  

In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/XRPUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/XRPUSDT__h5_model.joblib
[saved] features -> models/xgb/XRPUSDT__h5_feature_cols.json
[saved] feature importance -> models/xgb/XRPUSDT__h5_feature_importance.csv
[saved] metadata -> models/xgb/XRPUSDT__h5_meta.json
